# Statistical Profiling - NovaMart E-Commerce Dataset

This notebook performs statistical profiling to understand data distributions, detect outliers, and identify patterns.

## Why This Order Matters

We follow the **data flow of the business**:
1. **Core Transactions** → What is the business actually doing?
2. **Dimension Tables** → What are we selling, to whom, and through whom?
3. **Operational Data** → How well are we executing?
4. **Customer Experience** → What do customers think?
5. **Financial Impact** → Where is money being made or lost?
6. **Categorical Analysis** → What are the key segments?

In [ ]:
import os
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv

In [ ]:
load_dotenv()

conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema=os.getenv('SNOWFLAKE_SCHEMA'),
    role=os.getenv('SNOWFLAKE_ROLE')
)

---
## 1. Core Transactional Data

**WHY FIRST?** Orders and order items are the heartbeat of the business. Every downstream analysis (revenue, fulfillment, customer satisfaction) depends on understanding transaction patterns first.

**WHAT TO LOOK FOR:**
- Order value distribution → Are there outliers skewing averages?
- Shipping cost patterns → Basis for profitability analysis
- Quantity patterns → Impacts inventory planning

In [ ]:
# 1.1 ORDERS - The central fact table
query = """SELECT * FROM sales_raw."01_ORDERS" """
df_orders = pd.read_sql(query, conn)
df_orders.columns = df_orders.columns.str.lower()

print("=== 01_ORDERS: Numeric Distributions ===")
print(df_orders[['order_total', 'shipping_cost', 'total_weight_kg', 'num_items']].describe())

# Outlier detection using IQR method
print("\n=== Outlier Detection (IQR Method) ===")
for col in ['order_total', 'shipping_cost']:
    Q1, Q3 = df_orders[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = df_orders[(df_orders[col] < Q1 - 1.5*IQR) | (df_orders[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df_orders)*100:.1f}%)")

In [ ]:
# 1.2 ORDER_ITEMS - Line-item detail (granular view of transactions)
query = """SELECT * FROM sales_raw."02_ORDER_ITEMS" """
df_items = pd.read_sql(query, conn)
df_items.columns = df_items.columns.str.lower()

print("=== 02_ORDER_ITEMS: Numeric Distributions ===")
print(df_items[['quantity', 'unit_price', 'discount_pct', 'item_total', 'item_weight_kg']].describe())

print("\n=== Outlier Detection (IQR Method) ===")
for col in ['unit_price', 'item_total']:
    Q1, Q3 = df_items[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = df_items[(df_items[col] < Q1 - 1.5*IQR) | (df_items[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df_items)*100:.1f}%)")

---
## 2. Dimension Tables (Master Data)

**WHY SECOND?** Dimension tables define the "who, what, and through whom" of every transaction. Understanding these distributions helps you:
- Segment analysis properly
- Identify data quality issues in reference data
- Set realistic expectations for joins

**WHAT TO LOOK FOR:**
- Product pricing and margins → Profitability potential
- Customer lifetime value distribution → Segmentation basis
- Seller quality scores → Partner performance

In [ ]:
# 2.1 PRODUCTS - What we sell
query = """SELECT * FROM sales_raw."03_PRODUCTS" """
df_products = pd.read_sql(query, conn)
df_products.columns = df_products.columns.str.lower()

print("=== 03_PRODUCTS: Numeric Distributions ===")
print(df_products[['price', 'cost_price', 'weight_kg', 'length_cm', 'width_cm', 'height_cm']].describe())

# Profit margin analysis - CRITICAL for business understanding
df_products['margin_pct'] = ((df_products['price'] - df_products['cost_price']) / df_products['price'] * 100)
print("\n=== Profit Margin Distribution ===")
print(df_products['margin_pct'].describe())

In [ ]:
# 2.2 CUSTOMERS - Who buys from us
query = """SELECT * FROM sales_raw."04_CUSTOMERS" """
df_customers = pd.read_sql(query, conn)
df_customers.columns = df_customers.columns.str.lower()

print("=== 04_CUSTOMERS: Numeric Distributions ===")
print(df_customers[['lifetime_value_estimate']].describe())

# Customer segment distribution - key for targeting
print("\n=== Customer Segment Distribution ===")
print(df_customers['customer_segment'].value_counts())

In [ ]:
# 2.3 SELLERS - Who fulfills orders
query = """SELECT * FROM sales_raw."05_SELLERS" """
df_sellers = pd.read_sql(query, conn)
df_sellers.columns = df_sellers.columns.str.lower()

print("=== 05_SELLERS: Performance Metrics ===")
print(df_sellers[['avg_processing_hours', 'quality_score', 'total_products_listed', 'return_policy_days']].describe())

# Fulfillment type distribution
print("\n=== Fulfillment Type Distribution ===")
print(df_sellers['fulfillment_type'].value_counts())

---
## 3. Operational Data

**WHY THIRD?** Once you understand transactions and master data, you need to assess execution quality. This reveals:
- Fulfillment bottlenecks
- Inventory health
- Operational capacity issues

**WHAT TO LOOK FOR:**
- Days of supply → Stockout risk
- Stockout rates → Lost revenue indicator
- Event timing patterns → Process efficiency

In [ ]:
# 3.1 INVENTORY_SNAPSHOTS - Stock levels over time
query = """SELECT * FROM sales_raw."09_INVENTORY_SNAPSHOTS" """
df_inventory = pd.read_sql(query, conn)
df_inventory.columns = df_inventory.columns.str.lower()

print("=== 09_INVENTORY_SNAPSHOTS: Stock Metrics ===")
print(df_inventory[['quantity_on_hand', 'quantity_reserved', 'quantity_available', 'weekly_demand', 'days_of_supply']].describe())

# CRITICAL METRIC: Stockout frequency
stockout_pct = df_inventory['is_stockout'].mean() * 100
print(f"\n=== CRITICAL: Stockout Rate: {stockout_pct:.2f}% of snapshots ===")
print("(High stockout rate = lost revenue opportunity)")

---
## 4. Customer Experience Data

**WHY FOURTH?** Customer feedback is a leading indicator of future business health. Problems here predict:
- Churn risk
- Negative word-of-mouth
- Return rates

**WHAT TO LOOK FOR:**
- Rating distributions → Overall satisfaction
- Delivery vs product ratings → Root cause isolation
- Resolution times → Service quality

In [ ]:
# 4.1 CUSTOMER_REVIEWS - Voice of the customer
query = """SELECT * FROM sales_raw."08_CUSTOMER_REVIEWS" """
df_reviews = pd.read_sql(query, conn)
df_reviews.columns = df_reviews.columns.str.lower()

print("=== 08_CUSTOMER_REVIEWS: Rating Distributions ===")
print(df_reviews[['rating', 'delivery_rating', 'product_rating', 'days_late', 'helpful_votes']].describe())

# Rating distribution - key health metric
print("\n=== Overall Rating Distribution ===")
print(df_reviews['rating'].value_counts().sort_index())

# Late delivery impact
late_deliveries = (df_reviews['days_late'] > 0).mean() * 100
print(f"\n=== Late Delivery Rate: {late_deliveries:.1f}% of reviewed orders ===")

In [ ]:
# 4.2 SUPPORT_TICKETS - Customer issues and resolution
query = """SELECT * FROM sales_raw."11_SUPPORT_TICKETS" """
df_tickets = pd.read_sql(query, conn)
df_tickets.columns = df_tickets.columns.str.lower()

print("=== 11_SUPPORT_TICKETS: Resolution Metrics ===")
print(df_tickets[['resolution_hours', 'first_response_minutes', 'num_interactions', 'satisfaction_score']].describe())

# Issue type distribution - reveals problem areas
print("\n=== Issue Type Distribution ===")
print(df_tickets['issue_type'].value_counts())

---
## 5. Financial Impact Data

**WHY FIFTH?** After understanding operations and customer experience, we quantify the financial impact:
- Returns = direct revenue loss
- Marketing = customer acquisition cost
- Cancellations = demand that didn't convert

**WHAT TO LOOK FOR:**
- Refund amounts → Revenue leakage
- Return reasons → Actionable insights
- ROAS → Marketing efficiency

In [ ]:
# 5.1 RETURNS_CANCELLATIONS - Revenue leakage
query = """SELECT * FROM sales_raw."07_RETURNS_CANCELLATIONS" """
df_returns = pd.read_sql(query, conn)
df_returns.columns = df_returns.columns.str.lower()

print("=== 07_RETURNS_CANCELLATIONS: Refund Metrics ===")
print(df_returns[['refund_amount', 'restocking_fee']].describe())

# Return reasons - actionable insights
print("\n=== Return Reason Distribution (Actionable!) ===")
print(df_returns['return_reason'].value_counts())

# Total revenue leakage
total_refunds = df_returns['refund_amount'].sum()
print(f"\n=== Total Refund Amount: ${total_refunds:,.2f} ===")

In [ ]:
# 5.2 MARKETING_CAMPAIGNS - Customer acquisition cost
query = """SELECT * FROM sales_raw."10_MARKETING_CAMPAIGNS" """
df_marketing = pd.read_sql(query, conn)
df_marketing.columns = df_marketing.columns.str.lower()

print("=== 10_MARKETING_CAMPAIGNS: Performance Metrics ===")
print(df_marketing[['spend', 'impressions', 'clicks', 'conversions', 'revenue_attributed']].describe())

# ROAS distribution - marketing efficiency
print("\n=== ROAS (Return on Ad Spend) Distribution ===")
print(df_marketing['return_on_ad_spend'].describe())

# Channel performance summary
print("\n=== ROAS by Channel ===")
print(df_marketing.groupby('channel')['return_on_ad_spend'].mean().sort_values(ascending=False))

In [ ]:
# 5.3 CANCELLATIONS - Demand that didn't convert
query = """SELECT * FROM sales_raw."12_CANCELLATIONS" """
df_cancellations = pd.read_sql(query, conn)
df_cancellations.columns = df_cancellations.columns.str.lower()

print("=== 12_CANCELLATIONS: Lost Order Value ===")
print(df_cancellations[['was_order_total']].describe())

# Cancellation reasons - actionable insights
print("\n=== Cancellation Reason Distribution ===")
print(df_cancellations['cancellation_reason'].value_counts())

# Total lost revenue
total_cancelled = df_cancellations['was_order_total'].sum()
print(f"\n=== Total Cancelled Order Value: ${total_cancelled:,.2f} ===")

---
## 6. Categorical Distributions

**WHY LAST?** After understanding numeric distributions, we analyze categorical segments to:
- Identify dominant vs. niche segments
- Spot data quality issues (unexpected categories)
- Prepare for segmented analysis

In [ ]:
# 6.1 Order Categorical Distributions
print("=== Order Status Distribution ===")
print(df_orders['order_status'].value_counts())

print("\n=== Carrier Distribution ===")
print(df_orders['carrier'].value_counts())

print("\n=== Payment Method Distribution ===")
print(df_orders['payment_method'].value_counts())

print("\n=== Order Channel Distribution ===")
print(df_orders['order_channel'].value_counts())

print("\n=== Customer Region Distribution ===")
print(df_orders['customer_region'].value_counts())

In [ ]:
# 6.2 Product and Customer Categorical Distributions
print("=== Product Category Distribution ===")
print(df_products['category'].value_counts())

print("\n=== Acquisition Channel Distribution ===")
print(df_customers['acquisition_channel'].value_counts())

---
## Key Findings Summary

After running this notebook, you should document:

1. **Outliers identified** - Which need investigation vs. are legitimate?
2. **Distribution shapes** - Normal, skewed, bimodal?
3. **Critical metrics** - Stockout rate, late delivery %, refund totals
4. **Segment imbalances** - Any categories dominating or missing?
5. **Data quality flags** - Unexpected values, missing categories?

In [ ]:
conn.close()
print("Connection closed. Statistical profiling complete.")